# Working

**Request:** [T05] Billing vs ledger for the month

> New email from Sai Suram <sai@agents.agentstore.it.com>
> Subject: [T05] Billing vs ledger for the month
> Thread ID: AAQkADI1N2Y5MTE3LTE1MDctNGY0Yy1iYzQ5LWEzNmE5NzAyYzk4NQAQAGPdBYVlO31HrD-4zvuVJiA=
> 
> ## T

Each cell below is one run in the sandbox, in the order it happened, with whatever it printed and produced.


In [1]:
import pandas as pd
import io

billing_data = """Invoice,Amount
INV-001,1200.00
INV-002,850.00
INV-003,2300.00
INV-004,640.00
INV-005,1750.00"""

ledger_data = """Invoice,Amount
INV-001,1200.00
INV-002,850.00
INV-003,2030.00
INV-005,1750.00
INV-005,1750.00"""

billing_df = pd.read_csv(io.StringIO(billing_data))
ledger_df = pd.read_csv(io.StringIO(ledger_data))

# Group ledger data to handle duplicates
ledger_df_agg = ledger_df.groupby('Invoice')['Amount'].sum().reset_index()

# Merge dataframes for comparison
merged_df = pd.merge(billing_df, ledger_df_agg, on='Invoice', suffixes=('_Billing', '_Ledger'), how='outer')

# Identify differences
merged_df['Difference'] = merged_df['Amount_Billing'].fillna(0) - merged_df['Amount_Ledger'].fillna(0)

differences = merged_df[merged_df['Difference'] != 0]

# Find invoices missing from ledger (where Amount_Ledger is NaN)
invoices_missing_from_ledger = merged_df[merged_df['Amount_Ledger'].isna() & merged_df['Amount_Billing'].notna()]

# Find invoices missing from billing (where Amount_Billing is NaN) - should not happen if billing is source
invoices_missing_from_billing = merged_df[merged_df['Amount_Billing'].isna() & merged_df['Amount_Ledger'].notna()]

# Handle duplicate entries in ledger (already handled by groupby sum for now, but need to note if it was a distinct entry issue)
# For distinct duplicate detection, we would need to check original ledger_df

# Let's re-examine ledger_df for true duplicates that are not just aggregated differently
duplicate_ledger_entries = ledger_df[ledger_df.duplicated(subset=['Invoice'], keep=False)].sort_values(by='Invoice')

# Prepare output for the next step

# This part will be the content of the Excel file
output_df = merged_df.copy()
output_df['Status'] = 'Matched'
output_df.loc[output_df['Amount_Billing'].isna(), 'Status'] = 'Missing in Billing'
output_df.loc[output_df['Amount_Ledger'].isna(), 'Status'] = 'Missing in Ledger'
output_df.loc[output_df['Difference'] != 0, 'Status'] = 'Amount Mismatch'

# Add the duplicate entries from ledger explicitly for clarity
if not duplicate_ledger_entries.empty:
    duplicate_ledger_entries['Status'] = 'Duplicate in Ledger'
    duplicate_ledger_entries.rename(columns={'Amount': 'Amount_Ledger'}, inplace=True)
    duplicate_ledger_entries['Amount_Billing'] = pd.NA
    duplicate_ledger_entries['Difference'] = pd.NA
    output_df = pd.concat([output_df, duplicate_ledger_entries[['Invoice', 'Amount_Billing', 'Amount_Ledger', 'Difference', 'Status']]], ignore_index=True)


output_df.to_excel('/tmp/output/billing_ledger_reconciliation.xlsx', index=False)

print(output_df.to_json(orient='records'))


[{"Invoice":"INV-001","Amount_Billing":1200.0,"Amount_Ledger":1200.0,"Difference":0.0,"Status":"Matched"},{"Invoice":"INV-002","Amount_Billing":850.0,"Amount_Ledger":850.0,"Difference":0.0,"Status":"Matched"},{"Invoice":"INV-003","Amount_Billing":2300.0,"Amount_Ledger":2030.0,"Difference":270.0,"Status":"Amount Mismatch"},{"Invoice":"INV-004","Amount_Billing":640.0,"Amount_Ledger":null,"Difference":640.0,"Status":"Amount Mismatch"},{"Invoice":"INV-005","Amount_Billing":1750.0,"Amount_Ledger":3500.0,"Difference":-1750.0,"Status":"Amount Mismatch"},{"Invoice":"INV-005","Amount_Billing":null,"Amount_Ledger":1750.0,"Difference":null,"Status":"Duplicate in Ledger"},{"Invoice":"INV-005","Amount_Billing":null,"Amount_Ledger":1750.0,"Difference":null,"Status":"Duplicate in Ledger"}]



[files written: billing_ledger_reconciliation.xlsx]
